# 

## Import

In [12]:
import json
import os
import random

import numpy as np
import torch
from torch.utils.data import DataLoader

In [13]:
print(f"PyTorch 版本: {torch.__version__}")
print(f"CUDA 可用: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

PyTorch 版本: 2.13.0+cu130
CUDA 可用: True
GPU: NVIDIA GeForce RTX 5090


## Configuration

In [ ]:
# 全部可選波段（參考用，共 8 種）
ALL_BANDS = ["Green", "Red", "NIR", "SWIR", "SCL", "NDVI", "NDWI", "LSWI"]

# 自定義要使用的波段 ── 從 ALL_BANDS 中選取，順序即為輸出通道順序
# （SCL one-hot 暫時不使用，見 docs/Sentinel2_ModelA.md；要啟用時加回 "SCL" 即可）
CHOSEN_BANDS = ["Green", "Red", "NIR", "SWIR"]

# split_mode 控制本次執行跑哪個切分方式：
#   "time"    → Case1：以「灌水日期」分割（2023+2026 全部日期隨機切 6:2:2）
#   "spatial" → Case2：以「區塊」分割（11 個區塊 7:2:2）
# 可用環境變數 SPLIT_MODE 覆寫，方便用 nbconvert --execute 分別批次跑兩個 case
# 而不用手動改這個 cell。
SPLIT_MODE = os.environ.get("SPLIT_MODE", "time")
assert SPLIT_MODE in ("time", "spatial"), f"SPLIT_MODE 必須是 'time' 或 'spatial'，收到：{SPLIT_MODE!r}"

_BASE_OUTPUT_DIR = r"C:\Users\cathyhsu\Desktop\Sentinal2-ModelA-Irrigation\model_output"

# 模型名稱：決定 model_output/ 下的輸出資料夾與 TensorBoard run 名稱。
# 每個模型（base_model、M1_augmentation…）各自取名，互不覆蓋；
# 可用環境變數 MODEL_NAME 覆寫，方便批次跑多個模型。
MODEL_NAME = os.environ.get("MODEL_NAME", "base_model")

CONFIG = {
    # ── Path ──────────────────────────────────────────────────────────────
    "xy_dir":               r"D:\嘉璯\S2全台資料_Xy對應",  # 切割成 11 區域後的資料夾
    "output_dir":           os.path.join(_BASE_OUTPUT_DIR, MODEL_NAME),
    # run_name 自動取 output_dir 最後一段，不需另外設定
    # 切割結果與正規化統計量存於 split_ways/（見 CONFIG 建立後的 split_dir），
    # 與模型名稱無關，可跨模型重用

    # ── Split ─────────────────────────────────────────────────────────────
    "split_mode":           SPLIT_MODE,
    "seed":                 42,
    "lag":                  1,            # 每個 y 日期取「小於等於該 y 日期」中最大的 lag 筆 X 日期
    "time_train_ratio":     0.6,          # Case1：全部日期（2023+2026）train 比例
    "time_val_ratio":       0.2,          # Case1：valid 比例，其餘歸 test → 6:2:2
    "region_train_ratio":   0.6,          # Case2：11 個區塊 round(11*0.6)=7
    "region_val_ratio":     0.2,          # Case2：round(11*0.2)=2 → 7:2:2

    # ── Data ──────────────────────────────────────────────────────────────
    "generate_patch":       False,          # True=重新切 patch 並重算正規化統計量；
                                            # False 且 split_dir 已有對應 json 時直接重用
    "chosen_bands":         CHOSEN_BANDS,
    "nodata_value":         999,
    "scl_num_classes":      12,           # Sentinel-2 SCL 固定 12 類（one-hot 另加 1 類 nodata/異常值 → 12+1 通道）
    "patch_size":           256,
    "stride":               128,          # train patch stride（向右與向下相同；= patch_size 為非重疊，128 為 50% 重疊）
    "keep_remainder":       False,
    "nday":                 20,           # 時間差通道正規化分母（X 資料視窗上限 nday20）
    # 以下為 Dataset 篩選／採樣參數（依 train_info.json 的比例欄位，僅 train 使用；
    # 切割階段不做任何篩選）
    "filter_all_nodata":    True,         # 剔除整張皆為 nodata 的 patch（nodata_ratio == 1）
    "filter_no_water":      False,        # 剔除有效像素中無水體的 patch（water_ratio == 0）
    "cloud_rate_threshold": None,         # None=不篩選；e.g. 0.3=剔除 scl_cloud_ratio >30% 的 patch
    "no_water_keep_ratio":  0.25,         # 無灌水 patch 下採樣：有灌水 n 張全保留，
                                            # 無灌水（water_ratio==0）隨機保留 n×0.25 張（None=不下採樣）
    "augment":              False,        # M1：train 隨機翻轉＋90° 旋轉（X/y/scl_mask 同步變換）

    # ── Loader ────────────────────────────────────────────────────────────
    "batch_size":           4,
    "num_workers":          4,            # DataLoader worker 數（24 核實測 4 workers ≈ 194 batch/s；
                                            # 搭配 persistent_workers 避免 Windows spawn 每個 epoch 重啟）
    "pin_memory":           torch.cuda.is_available(),
    "oversample_water":     False,        # M5：WeightedRandomSampler 提高高灌水 patch 抽樣機率
    "oversample_factor":    10,           # oversampling 權重 = 1 + factor × water_ratio

    # ── Model ─────────────────────────────────────────────────────────────
    "encoder_name":         "resnet34",

    # ── Loss ──────────────────────────────────────────────────────────────
    "pos_weight":           1.0,
    "bce_weight":           0.5,
    "dice_weight":          0.5,
    "scl_mask":             False,         # True=遮蔽 SCL∈{3,8,9}（雲影/中機率雲/高機率雲）的 pixel

    # ── Train ─────────────────────────────────────────────────────────────
    # EPOCHS/PATIENCE 環境變數可覆寫，方便先用極小值跑一次 nbconvert 驗證整條
    # pipeline 沒有錯誤，確認無誤後再用預設值（100/10）跑正式訓練。
    "epochs":               int(os.environ.get("EPOCHS", 100)),
    "patience":             int(os.environ.get("PATIENCE", 10)),
    "lr":                   1e-4,
    "weight_decay":         1e-5,
    "use_scheduler":        False,        # M4：ReduceLROnPlateau（監測 val_loss，停滯時降 lr）
    "scheduler_factor":     0.5,          # lr 調降倍率
    "scheduler_patience":   3,            # 連續幾個 epoch 未改善才調降
    "device":               "cuda" if torch.cuda.is_available() else "cpu",
}

# 切割結果資料夾：不跟隨模型名稱，依「切割方式 + patch_size + stride + keep_remainder」
# 命名，集中存於 split_ways/ 下（train/valid/test_info.json 與 norm_stats.json），
# 各模型透過上面這組參數選擇要使用的切割結果，可跨模型重用
CONFIG["split_dir"] = os.path.join(
    os.path.dirname(_BASE_OUTPUT_DIR), "split_ways",
    f"{SPLIT_MODE}_p{CONFIG['patch_size']}_s{CONFIG['stride']}_keep{CONFIG['keep_remainder']}",
)

In [15]:
def set_seed(seed: int = 42) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(CONFIG["seed"])
os.makedirs(CONFIG["output_dir"], exist_ok=True)

cfg_save = {k: v for k, v in CONFIG.items() if not isinstance(v, bool)}
with open(os.path.join(CONFIG["output_dir"], "config.json"), "w", encoding="utf-8") as f:
    json.dump(cfg_save, f, indent=4, ensure_ascii=False)
print("✓ config.json 已寫入")
print(f"  split_mode : {CONFIG['split_mode']}")
print(f"  output_dir : {CONFIG['output_dir']}")

✓ config.json 已寫入
  split_mode : time
  output_dir : C:\Users\cathyhsu\Desktop\Sentinal2-ModelA-Irrigation\model_output\case1_time_split


## Train-Valid-Test Split

### 使用資料範圍

#### Case1. 以「灌水日期」分割
2023 第 1、2 期作與 2026 第 1 期作的全部灌水日期，用固定 seed 隨機打亂後分成 6:2:2。

#### Case2. 以「區塊」分割
共有 11 個區塊，命名為 N1, N2, EN, E1, E2, E3, W1, W2, W3, W4, W5，分成 7:2:2（round(11×0.6)=7、round(11×0.2)=2）。


In [16]:
from src.preprocessing import Preprocess

# Case1（時間切分）與 Case2（空間切分）共用下游變數名稱：
#   train_dates_y / val_dates_y / test_dates_y → y 的日期
#   train_dates_X / val_dates_X / test_dates_X → 與 *_dates_y 一一對應的 X 日期
#       （每個 y 日期取「小於等於該 y 日期」中最大的 lag 筆 X 日期，lag 見 CONFIG）
#   train_regions / val_regions / test_regions → 要篩選的區塊（None = 不篩選，全部區塊）
train_regions = val_regions = test_regions = None

if CONFIG["split_mode"] == "time":
    # 1. 日期切分（Case1：以「灌水日期」分割，2023+2026 全部日期隨機切 6:2:2）
    (train_dates_X, train_dates_y,
     val_dates_X,   val_dates_y,
     test_dates_X,  test_dates_y,
     train_files, val_files, test_files) = Preprocess.train_valid_test_split_by_date(
        base_dir=CONFIG["xy_dir"],
        train_ratio=CONFIG["time_train_ratio"],
        val_ratio=CONFIG["time_val_ratio"],
        seed=CONFIG["seed"],
        lag=CONFIG["lag"],
    )
    print(f"✓ 日期切分（y）  train={train_dates_y}  val={val_dates_y}  test={test_dates_y}")
    print(f"✓ X 配對（lag={CONFIG['lag']}）")
    for split_name, ys, xs in [("train", train_dates_y, train_dates_X),
                               ("val",   val_dates_y,   val_dates_X),
                               ("test",  test_dates_y,  test_dates_X)]:
        print(f"  {split_name}: " + "  ".join(f"{y}←{x}" for y, x in zip(ys, xs)))
    print(f"✓ 檔案切分  train_files={len(train_files)} 個  val_files={len(val_files)} 個  test_files={len(test_files)} 個")
    print(f"  train_files 範例: {train_files[:5]}")
    print(f"  val_files   範例: {val_files[:5]}")
    print(f"  test_files  範例: {test_files[:5]}")
else:
    print(f"split_mode='{CONFIG['split_mode']}'，略過時間切分（見下一個 cell 的空間切分）")

✓ 日期切分（y）  train=['20230130', '20230306', '20230311', '20230709', '20230714', '20230716', '20230719', '20230721', '20230724', '20230725', '20230731', '20230814', '20230824', '20260119', '20260129', '20260203', '20260213', '20260223', '20260320', '20260414']  val=['20230209', '20230219', '20230301', '20230802', '20230812', '20260104', '20260330']  test=['20230204', '20230316', '20230704', '20260124', '20260315', '20260419']
✓ X 配對（lag=1）
  train: 20230130←['20230130']  20230306←['20230306']  20230311←['20230311']  20230709←['20230709']  20230714←['20230714']  20230716←['20230714']  20230719←['20230719']  20230721←['20230719']  20230724←['20230724']  20230725←['20230724']  20230731←['20230729']  20230814←['20230813']  20230824←['20230823']  20260119←['20260119']  20260129←['20260129']  20260203←['20260129']  20260213←['20260213']  20260223←['20260223']  20260320←['20260320']  20260414←['20260414']
  val: 20230209←['20230209']  20230219←['20230219']  20230301←['20230301']  20230802←['2023

In [17]:
from pathlib import Path
from src.preprocessing import Preprocess

if CONFIG["split_mode"] == "spatial":
    # 2. 區塊切分（Case2：以「區塊」分割，train/valid/test 共用「全部日期」，
    #    只依區塊歸屬做切分，避免同一區塊同時出現在不同 split 造成空間洩漏）
    train_regions, val_regions, test_regions, train_files, val_files, test_files = Preprocess.split_regions_train_valid_test(
        base_dir=CONFIG["xy_dir"],
        train_ratio=CONFIG["region_train_ratio"],
        val_ratio=CONFIG["region_val_ratio"],
        seed=CONFIG["seed"],
    )
    train_dates_y = val_dates_y = test_dates_y = sorted(
        p.name for p in (Path(CONFIG["xy_dir"]) / "y").iterdir() if p.is_dir()
    )
    # X 配對：每個 y 日期取「小於等於該 y 日期」中最大的 lag 筆 X 日期
    train_dates_X = val_dates_X = test_dates_X = Preprocess.match_x_dates(
        CONFIG["xy_dir"], train_dates_y, lag=CONFIG["lag"],
    )
    print(f"✓ 區塊切分  train={train_regions}  val={val_regions}  test={test_regions}")
    print(f"✓ 共用日期數（train=val=test 皆為全部日期，只依區塊篩選）：{len(train_dates_y)}")
    print(f"✓ X 配對（lag={CONFIG['lag']}）  " + "  ".join(f"{y}←{x}" for y, x in zip(train_dates_y[:3], train_dates_X[:3])) + " ...")
    print(f"✓ 檔案切分  train_files={len(train_files)} 個  val_files={len(val_files)} 個  test_files={len(test_files)} 個")
    print(f"  train_files 範例: {train_files[:5]}")
    print(f"  val_files   範例: {val_files[:5]}")
    print(f"  test_files  範例: {test_files[:5]}")
else:
    print(f"split_mode='{CONFIG['split_mode']}'，略過空間切分")

split_mode='time'，略過空間切分


## Normalization

In [18]:
from src.preprocessing import Preprocess

# 標準化參數（只用 train；統計對象是 X，因此使用 train_dates_X 配對到的
# X 日期攤平去重後的清單，來源為切割成 11 區域後的 "xy對應" 資料夾，
# 只計算前 4 個原始波段 Green/Red/NIR/SWIR）
#
# 先用 train 統計出的 1st~99th 百分位數做離群值裁切，取代寫死的 (0, 10000)，
# clip 邊界與 mean/std 皆只能用 train 算，避免 val/test 洩漏。
# 統計時以 y/20230130（mask_date）各區塊中 999 的位置遮蔽 X（各日期同一
# 區塊的 999 位置相同），並剔除 X 中的 65535 nodata；X 本身的 999 可能為
# 有效值，不剔除。
#
# 計算結果（clip_min/clip_max、mean/std）存至 split_dir/norm_stats.json
#（與切割結果同資料夾，跨模型重用）；
# generate_patch=False 且檔案已存在時直接讀取、不重算（與切割結果的重用邏輯一致）。
_RAW_BANDS = ["Green", "Red", "NIR", "SWIR"]
_train_x_dates = sorted({d for lags in train_dates_X for d in lags})

_norm_stats_path = os.path.join(CONFIG["split_dir"], "norm_stats.json")

if not CONFIG["generate_patch"] and os.path.exists(_norm_stats_path):
    with open(_norm_stats_path, encoding="utf-8") as f:
        _stats = json.load(f)
    clip_min = np.array(_stats["clip_min"], dtype=np.float32)
    clip_max = np.array(_stats["clip_max"], dtype=np.float32)
    mean     = np.array(_stats["mean"], dtype=np.float32)
    std      = np.array(_stats["std"], dtype=np.float32)
    print(f"✓ 已讀取正規化統計量：{_norm_stats_path}")
else:
    clip_min, clip_max = Preprocess.compute_band_percentiles(
        base_dir=CONFIG["xy_dir"],
        dates=_train_x_dates,
        regions=train_regions,
        numeric_bands=_RAW_BANDS,
        lower=1.0,
        upper=99.0,
        mask_date="20230130",
    )
    mean, std = Preprocess.compute_band_mean_std(
        base_dir=CONFIG["xy_dir"],
        dates=_train_x_dates,
        regions=train_regions,
        numeric_bands=_RAW_BANDS,
        clip_min=clip_min,
        clip_max=clip_max,
        mask_date="20230130",
    )
    os.makedirs(os.path.dirname(_norm_stats_path), exist_ok=True)
    with open(_norm_stats_path, "w", encoding="utf-8") as f:
        json.dump({
            "bands": _RAW_BANDS,
            "clip_percentiles": [1.0, 99.0],
            "clip_min": np.asarray(clip_min, dtype=float).tolist(),
            "clip_max": np.asarray(clip_max, dtype=float).tolist(),
            "mean":     np.asarray(mean, dtype=float).tolist(),
            "std":      np.asarray(std, dtype=float).tolist(),
        }, f, ensure_ascii=False, indent=2)
    print(f"✓ 正規化統計量已儲存：{_norm_stats_path}")

print(f"  百分位裁切邊界  clip_min={clip_min}  clip_max={clip_max}")
print(f"  標準化參數      mean={mean}  std={std}")

KeyboardInterrupt: 

## Cut Patches

In [ ]:
from src.preprocessing import data_cut

# 切割 patch（train / valid / test）：三個 split 的 json 集中輸出到
# CONFIG["split_dir"]（split_ways/{切割方式}_p{patch}_s{stride}_keep{bool}/，
# 與模型名稱無關、跨模型重用），檔名依 mode 固定：
#   train_info.json / valid_info.json / test_info.json
# X 來源日期使用 *_dates_X（match_x_dates() 配對結果，lag>1 時取最近一筆）。
# 切割階段不做任何篩選、不存 patch .npy；train 的每個 patch 條目記錄
# water_ratio / scl_cloud_ratio / nodata_ratio（供宣告 Dataset 時篩選）。
def _get_or_cut(mode, dates, dates_X, regions):
    existing = os.path.join(CONFIG["split_dir"], f"{mode}_info.json")
    if not CONFIG["generate_patch"] and os.path.exists(existing):
        return existing
    return data_cut.cut_xy_patches(
        xy_dir=CONFIG["xy_dir"], dates=dates, dates_X=dates_X, regions=regions, mode=mode,
        out_dir=CONFIG["split_dir"],
        patch_size=CONFIG["patch_size"],
        stride=CONFIG["stride"],
        keep_remainder=CONFIG["keep_remainder"] if mode == "train" else True,
        nodata_value=CONFIG["nodata_value"],
    )

train_info = _get_or_cut("train", train_dates_y, train_dates_X, train_regions)
val_info   = _get_or_cut("valid", val_dates_y,   val_dates_X,   val_regions)
test_info  = _get_or_cut("test",  test_dates_y,  test_dates_X,  test_regions)

print(f"✓ Patch 切割完成")
print(f"  train_info={train_info}")
print(f"  val_info  ={val_info}")
print(f"  test_info ={test_info}")

── 處理 20230130/E1 ──
  共切出 1444 張 patches
── 處理 20230130/E2 ──
  共切出 1444 張 patches
── 處理 20230130/E3 ──
  共切出 1444 張 patches
── 處理 20230130/EN ──
  共切出 1444 張 patches
── 處理 20230130/N1 ──
  共切出 2166 張 patches
── 處理 20230130/N2 ──
  共切出 2166 張 patches
── 處理 20230130/W1 ──
  共切出 2166 張 patches
── 處理 20230130/W2 ──
  共切出 2926 張 patches
── 處理 20230130/W3 ──
  共切出 2166 張 patches
── 處理 20230130/W4 ──
  共切出 2166 張 patches
── 處理 20230130/W5 ──
  共切出 1444 張 patches
── 處理 20230306/E1 ──
  共切出 1444 張 patches
── 處理 20230306/E2 ──
  共切出 1444 張 patches
── 處理 20230306/E3 ──
  共切出 1444 張 patches
── 處理 20230306/EN ──
  共切出 1444 張 patches
── 處理 20230306/N1 ──
  共切出 2166 張 patches
── 處理 20230306/N2 ──
  共切出 2166 張 patches
── 處理 20230306/W1 ──
  共切出 2166 張 patches
── 處理 20230306/W2 ──
  共切出 2926 張 patches
── 處理 20230306/W3 ──
  共切出 2166 張 patches
── 處理 20230306/W4 ──
  共切出 2166 張 patches
── 處理 20230306/W5 ──
  共切出 1444 張 patches
── 處理 20230311/E1 ──
  共切出 1444 張 patches
── 處理 20230311/E2 ──
  共切出 1444 張 

## Dataset

In [ ]:
from src.dataset import Dataset

# Dataset：train/valid/test 一律依 *_info.json 的座標從 xy_dir 原始大圖
# memmap 即時切割（必須傳入 xy_dir）。
# 每個 patch 的最後一個通道為時間差輔助通道（(y−X) 天數差 / nday 的常數平面），
# 已取代原本的 nodata mask 通道。
# train 專屬篩選（filter_all_nodata / filter_no_water / cloud_rate_threshold）
# 與無灌水 patch 下採樣（no_water_keep_ratio：有灌水 n 張全保留、
# 無灌水隨機留 n×ratio 張）在此依 train_info.json 的比例欄位進行
# （切割階段不做篩選）。
train_dataset = Dataset.OneSliceDataset(
    info_json=train_info, dates=train_dates_y,
    features=CONFIG["chosen_bands"],
    mean=mean, std=std, clip_min=clip_min, clip_max=clip_max,
    scl_num_classes=CONFIG["scl_num_classes"],
    nodata_value=CONFIG["nodata_value"],
    xy_dir=CONFIG["xy_dir"],
    nday=CONFIG["nday"],
    filter_all_nodata=CONFIG["filter_all_nodata"],
    filter_no_water=CONFIG["filter_no_water"],
    cloud_rate_threshold=CONFIG["cloud_rate_threshold"],
    no_water_keep_ratio=CONFIG["no_water_keep_ratio"],
    seed=CONFIG["seed"],
    augment=CONFIG["augment"],
)
val_dataset = Dataset.OneSliceDataset(
    info_json=val_info, dates=val_dates_y,
    features=CONFIG["chosen_bands"],
    mean=mean, std=std, clip_min=clip_min, clip_max=clip_max,
    scl_num_classes=CONFIG["scl_num_classes"],
    nodata_value=CONFIG["nodata_value"],
    xy_dir=CONFIG["xy_dir"],
    nday=CONFIG["nday"],
)
test_dataset = Dataset.OneSliceDataset(
    info_json=test_info, dates=test_dates_y,
    features=CONFIG["chosen_bands"],
    mean=mean, std=std, clip_min=clip_min, clip_max=clip_max,
    scl_num_classes=CONFIG["scl_num_classes"],
    nodata_value=CONFIG["nodata_value"],
    xy_dir=CONFIG["xy_dir"],
    nday=CONFIG["nday"],
)
print(f"✓ Dataset  train={len(train_dataset)}  val={len(val_dataset)}  test={len(test_dataset)}")
print(f"  每個 patch 通道數 = {train_dataset.num_channels}")

print(f"\nTrain: {len(train_dataset)} patches")
print(f"Valid: {len(val_dataset)} patches")
print(f"Test:  {len(test_dataset)} patches")

X_sample, y_sample, _ = train_dataset[0]
print(f"\nX shape: {tuple(X_sample.shape)}  dtype={X_sample.dtype}")
print(f"y shape: {tuple(y_sample.shape)}  dtype={y_sample.dtype}")

## EDA

In [ ]:
import io
from contextlib import redirect_stdout
from src.dataset import eda

# EDA：所有訓練資料「整體」統計（不分日期／區塊；直接用 train_info.json 已算好的
# 比例欄位，kept_indices=train_dataset.kept_indices → 只統計篩選＋下採樣後
# 實際用於訓練的 patch）。欄位：
#   total / all_nodata（全 nodata）/ all_zero（無灌水）/ have_one（含灌水）
#   / water_ratio_%_mean（平均灌水率）/ scl_cloud_ratio_%_mean（平均遮蔽率）
# 結果暫存於 eda_text（於下一個 cell 存檔，最後由「TensorBoard 匯整」寫入 TEXT tab）
eda_text = ""
header = (
    "\n══════════════════════════════════════════════\n"
    "  Train patch 整體統計（篩選後）\n"
    "══════════════════════════════════════════════\n"
)
print(header, end="")
buf = io.StringIO()
with redirect_stdout(buf):
    eda.count_zero_nodata_patches(
        info_json=train_info, dates=train_dates_y,
        kept_indices=train_dataset.kept_indices,
    )
result = buf.getvalue()
print(result)
eda_text += header + result + "\n"

In [ ]:
from src.dataset import eda

# EDA：所有訓練資料「整體」的灌水率與遮蔽率分布（各一張直方圖，不分日期／區塊；
# kept_indices → 只統計篩選＋下採樣後實際用於訓練的 patch）
df = eda.compute_irrigation_cloud_cover_rate(
    info_json=train_info,
    dates=train_dates_y,
    kept_indices=train_dataset.kept_indices,
)
eda.plot_irrigation_cloud_distribution(
    df,
    save_path=os.path.join(CONFIG["output_dir"], "eda_irrigation_cloud_distribution.png"),
)

# EDA 統計存檔（TensorBoard 統一由最後的「TensorBoard 匯整」cell 依此檔案寫入 TEXT tab）
with open(os.path.join(CONFIG["output_dir"], "eda.txt"), "w", encoding="utf-8") as f:
    f.write(eda_text)
print("✓ EDA 統計已儲存至 output_dir/eda.txt")

## Dataloader

In [ ]:
# 4. DataLoader
# val/test 不需打亂；train 預設 shuffle=True，oversample_water=True（M5）時改用
# WeightedRandomSampler（權重 = 1 + oversample_factor × water_ratio，提高高灌水
# 比例 patch 被抽樣到的機率；sampler 與 shuffle 互斥）。
# persistent_workers：num_workers > 0 時保留 worker 行程，避免 Windows spawn
# 每個 epoch 重新啟動 worker 的成本。
_persistent = CONFIG["num_workers"] > 0

_train_sampler = None
if CONFIG["oversample_water"]:
    _weights = torch.tensor(
        [1.0 + CONFIG["oversample_factor"] * (s["water_ratio"] or 0.0)
         for s in train_dataset.samples],
        dtype=torch.double,
    )
    _train_sampler = torch.utils.data.WeightedRandomSampler(
        _weights, num_samples=len(train_dataset), replacement=True,
    )
    print(f"✓ 高灌水 patch oversampling 啟用（factor={CONFIG['oversample_factor']}）")

train_loader = DataLoader(
    train_dataset,
    batch_size=CONFIG["batch_size"],
    shuffle=(_train_sampler is None),
    sampler=_train_sampler,
    num_workers=CONFIG["num_workers"],
    pin_memory=CONFIG["pin_memory"],
    persistent_workers=_persistent,
)
val_loader = DataLoader(
    val_dataset,
    batch_size=CONFIG["batch_size"],
    shuffle=False,
    num_workers=CONFIG["num_workers"],
    pin_memory=CONFIG["pin_memory"],
    persistent_workers=_persistent,
)
test_loader = DataLoader(
    test_dataset,
    batch_size=CONFIG["batch_size"],
    shuffle=False,
    num_workers=CONFIG["num_workers"],
    pin_memory=CONFIG["pin_memory"],
    persistent_workers=_persistent,
)
print(f"✓ DataLoader  train={len(train_loader)} batches  val={len(val_loader)} batches  test={len(test_loader)} batches")

✓ DataLoader  train=3974 batches  val=2900 batches  test=1450 batches


## Model

In [ ]:
from src.model import Model

# 5. 建立模型、Loss、Optimizer
device = CONFIG["device"]

model = Model.build_model(
    in_channels=train_dataset.num_channels,
    encoder_name=CONFIG["encoder_name"],
).to(device)

criterion = Model.MaskedDiceBCELoss(
    ignore_index=CONFIG["nodata_value"],
    bce_weight=CONFIG["bce_weight"],
    dice_weight=CONFIG["dice_weight"],
    pos_weight=CONFIG["pos_weight"],
    use_scl_mask=CONFIG["scl_mask"],
)

# 依規劃文件（Sentinel2_ModelA.md）使用 AdamW
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=CONFIG["lr"],
    weight_decay=CONFIG["weight_decay"],
)

print(f"✓ 模型建立完成  device={device}  in_channels={train_dataset.num_channels}")

✓ 模型建立完成  device=cuda  in_channels=5


## Training

In [15]:
from src.training import TrainingFuncs

# 6. 訓練（含 early stopping 與 checkpoint）
# 每個 epoch 的 train/valid 結果（loss/acc/f1）與 best_val_* 會即時覆寫至
# output_dir/train_history.json（中斷也保留）；TensorBoard 統一由最後的
# 「TensorBoard 匯整」cell 依該檔案重建，訓練中不直接寫入。
history = TrainingFuncs.training(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    criterion=criterion,
    optimizer=optimizer,
    config=CONFIG,
    device=device,
)

[Epoch 001/100] loss 0.4334/0.4678  f1 0.6594/0.7478
  ✓ 已儲存最佳模型（val_loss=0.4678）


[Epoch 002/100] loss 0.3985/0.4740  f1 0.7028/0.7469
  ⚠ 未改善 (1/10)


[Epoch 003/100] loss 0.3888/0.4607  f1 0.7125/0.7561
  ✓ 已儲存最佳模型（val_loss=0.4607）


[Epoch 004/100] loss 0.3805/0.4665  f1 0.7179/0.7524
  ⚠ 未改善 (1/10)


[Epoch 005/100] loss 0.3721/0.4589  f1 0.7237/0.7622
  ✓ 已儲存最佳模型（val_loss=0.4589）


...

[Epoch 012/100] loss 0.3410/0.4474  f1 0.7575/0.7727
  ✓ 已儲存最佳模型（val_loss=0.4474）


[Epoch 013/100] loss 0.3365/0.4575  f1 0.7614/0.7862
  ⚠ 未改善 (1/10)


[Epoch 014/100] loss 0.3340/0.4569  f1 0.7612/0.7700
  ⚠ 未改善 (2/10)


[Epoch 015/100] loss 0.3257/0.4768  f1 0.7665/0.7880
  ⚠ 未改善 (3/10)


KeyboardInterrupt: 

## Testing

In [ ]:
from src.training import TrainingFuncs

# 7. 載入最佳模型，在測試集評估，結果存成 output_dir/test_metrics.json
# （acc / precision / recall / f1 / IoU + confusion matrix；不輸出任何預測檔案，
# TensorBoard 統一由「TensorBoard 匯整」cell 依該檔案重建）
checkpoint = torch.load(CONFIG['output_dir'] + "/best_model.pt", map_location=device)
model.load_state_dict(checkpoint["model_state_dict"])

test_metrics_all = TrainingFuncs.evaluating_all(
    model, test_loader, device,
    ignore_index=CONFIG["nodata_value"],
)
print(f"✅ evaluating_all 完成：{test_metrics_all}")

_test_metrics_path = os.path.join(CONFIG["output_dir"], "test_metrics.json")
with open(_test_metrics_path, "w", encoding="utf-8") as f:
    json.dump(test_metrics_all, f, ensure_ascii=False, indent=2)
print(f"✓ 測試指標已儲存：{_test_metrics_path}")

## TensorBoard 匯整

In [ ]:
import sys
import glob
import shutil
from tensorboardX import SummaryWriter
from torch.utils.tensorboard import SummaryWriter as _TorchSW
from torch.utils.tensorboard.writer import hparams as _make_hparam_proto
from tensorboard.plugins.hparams.plugin_data_pb2 import HParamsPluginData
from PIL import Image
import torchvision.transforms.functional as TF

# ── TensorBoard 匯整 ──────────────────────────────────────────────────────
# 讀取 output_dir 中已儲存的結果檔，一次寫入單一 run（tb_logs/{run_name}）：
#   train_history.json    → 訓練曲線（Loss/Accuracy/F1，train/valid）＋ best_val_*
#   test_metrics.json     → 測試指標（acc/precision/recall/f1/IoU）
#   eda.txt               → TEXT tab
#   confusion_map_*.png   → IMAGES tab（由後面的「預測結果視覺化」cell 產生；
#                           畫完後重跑本 cell 才會納入）
# 訓練與測試放在同一個 run；本 cell 可重複執行——每次整個 run 清掉重建，
# 資料來源都是 output_dir 的檔案，與 kernel 狀態、是否中斷過無關。
run_name   = os.path.basename(CONFIG["output_dir"])
tb_root    = os.path.join(os.path.dirname(CONFIG["output_dir"]), "tb_logs")
tb_log_dir = os.path.join(tb_root, run_name)

if os.path.exists(tb_log_dir):
    shutil.rmtree(tb_log_dir)
    print(f"  舊 run 已清除：{tb_log_dir}")
# 清掉舊版流程留下的獨立 test run（現已合併進同一個 run）
_legacy_test = os.path.join(tb_root, f"{run_name}_test")
if os.path.exists(_legacy_test):
    shutil.rmtree(_legacy_test)
    print(f"  舊版獨立 test run 已清除：{_legacy_test}")
os.makedirs(tb_log_dir)

writer = SummaryWriter(log_dir=tb_log_dir)

# 1. 訓練曲線
_hist_path = os.path.join(CONFIG["output_dir"], "train_history.json")
history = None
if os.path.exists(_hist_path):
    with open(_hist_path, encoding="utf-8") as f:
        history = json.load(f)
    for i in range(len(history["train_loss"])):
        ep = i + 1
        writer.add_scalars("Loss",     {"train": history["train_loss"][i], "valid": history["val_loss"][i]}, ep)
        writer.add_scalars("Accuracy", {"train": history["train_acc"][i],  "valid": history["val_acc"][i]},  ep)
        writer.add_scalars("F1",       {"train": history["train_f1"][i],   "valid": history["val_f1"][i]},   ep)
    print(f"✓ 訓練曲線已寫入（{len(history['train_loss'])} epochs）")
else:
    print(f"[略過] 找不到 train_history.json")

# 2. EDA 統計（TEXT tab）
_eda_path = os.path.join(CONFIG["output_dir"], "eda.txt")
if os.path.exists(_eda_path):
    with open(_eda_path, encoding="utf-8") as f:
        writer.add_text("EDA", "```\n" + f.read() + "```", global_step=0)
    print("✓ EDA 統計已寫入 TEXT tab")

# 3. Confusion maps（IMAGES tab）
_pngs = sorted(glob.glob(os.path.join(CONFIG["output_dir"], "confusion_map_*.png")))
for _png in _pngs:
    _name = os.path.splitext(os.path.basename(_png))[0]
    writer.add_image(f"Test/{_name}", TF.to_tensor(Image.open(_png).convert("RGB")), global_step=0)
if _pngs:
    print(f"✓ Confusion maps 已寫入 IMAGES tab（{len(_pngs)} 張）")

writer.close()

# 4. HPARAMS（超參數 + best_val_* + 測試指標）
_metrics = {}
if history is not None:
    _metrics.update({
        "best_val_loss": float(history["best_val_loss"]),
        "best_val_acc":  float(history["best_val_acc"]),
        "best_val_f1":   float(history["best_val_f1"]),
        "best_epoch":    float(history["best_epoch"]),
    })
_test_path = os.path.join(CONFIG["output_dir"], "test_metrics.json")
if os.path.exists(_test_path):
    with open(_test_path, encoding="utf-8") as f:
        _tm = json.load(f)
    _metrics.update({
        "test_acc":       float(_tm["accuracy"]),
        "test_precision": float(_tm["precision"]),
        "test_recall":    float(_tm["recall"]),
        "test_f1":        float(_tm["f1"]),
        "test_IoU":       float(_tm["iou"]),
    })
else:
    print(f"[略過] 找不到 test_metrics.json（HPARAMS 只含訓練端指標）")

if _metrics:
    # cloud_rate_threshold / no_water_keep_ratio 為 None 時以 -1.0 代表「不篩選／不下採樣」
    _hparams = {
        "chosen_bands":         str(CONFIG["chosen_bands"]),
        "patch_size":           int(CONFIG["patch_size"]),
        "filter_all_nodata":    str(CONFIG["filter_all_nodata"]),
        "filter_no_water":      str(CONFIG["filter_no_water"]),
        "cloud_rate_threshold": float(CONFIG["cloud_rate_threshold"]) if CONFIG["cloud_rate_threshold"] is not None else -1.0,
        "no_water_keep_ratio":  float(CONFIG["no_water_keep_ratio"]) if CONFIG["no_water_keep_ratio"] is not None else -1.0,
        "scl_mask":             str(CONFIG["scl_mask"]),
        "pos_weight":           float(CONFIG["pos_weight"]),
        "batch_size":           int(CONFIG["batch_size"]),
        "lr":                   float(CONFIG["lr"]),
        "weight_decay":         float(CONFIG["weight_decay"]),
        "augment":              str(CONFIG["augment"]),
        "use_scheduler":        str(CONFIG["use_scheduler"]),
        "oversample_water":     str(CONFIG["oversample_water"]),
    }
    exp, ssi, sei = _make_hparam_proto(_hparams, _metrics)

    # 修正 TensorBoard 在 Windows 的 bug：metric group 為空字串時，後端用
    # os.path.join 組 run 名稱會多出結尾反斜線、查不到 metric scalars。
    # 改將 group 設為 "metrics"，並把指標 scalars 寫到 {run}/metrics 子資料夾。
    _pd = HParamsPluginData.FromString(exp.value[0].metadata.plugin_data.content)
    for _mi in _pd.experiment.metric_infos:
        _mi.name.group = "metrics"
    exp.value[0].metadata.plugin_data.content = _pd.SerializeToString()

    _hw = _TorchSW(log_dir=tb_log_dir)
    _hw.file_writer.add_summary(exp)
    _hw.file_writer.add_summary(ssi)
    _hw.file_writer.add_summary(sei)
    _hw.close()

    _hm = _TorchSW(log_dir=os.path.join(tb_log_dir, "metrics"))
    for k, v in _metrics.items():
        _hm.add_scalar(k, v, global_step=0)
    _hm.close()
    print(f"✓ HPARAMS 已寫入（{len(_metrics)} 個指標）")

print(f"\n✓ TensorBoard 匯整完成：{tb_log_dir}")
print(f"\n開啟 TensorBoard：")
print(f'  "{sys.executable}" -m tensorboard.main --logdir "{tb_root}"')
print(f"  → 在瀏覽器開啟 http://localhost:6006")

✓ TensorBoard writer 已關閉

開啟 TensorBoard（顯示所有 runs）：
  c:\Python311\python.exe -m tensorboard.main --logdir "C:\Users\cathyhsu\Desktop\Geospatial predictive modeling\data\S2訓練資料\model_output\tb_logs"
  → 在瀏覽器開啟 http://localhost:6006


## 預測結果視覺化（TP/TN/FP/FN map）

In [ ]:
import numpy as np
import torch
from src.visualization import myplot

# ── 指定 (date, region) 的 confusion map（即時推論，不落地 test_pred/test_gt）──
# 在 TARGETS 填入要輸出的組合（需存在於 test set）；留空執行會列出所有可用組合。
# 預測：載入 best_model 後，對該區域的 test patches 當場推論，
#       依 test_info.json 的座標重建成完整圖（nodata 位置蓋回 999）
# GT：  直接讀 xy_dir 的原始 y 大圖
# 圖片存成 output_dir/confusion_map_{date}_{region}.png；
# 若要把圖寫入 TensorBoard IMAGES tab，畫完後重跑一次上面的「TensorBoard 匯整」cell。
TARGETS = [
    # ("20230306", "N1"),
]

available = sorted({(s["date"], s["region"]) for s in test_dataset.samples})
if not TARGETS:
    print("TARGETS 為空。test set 可用的 (date, region) 組合如下，請挑選填入 TARGETS：")
    for d, r in available:
        print(f"  ({d!r}, {r!r})")

if TARGETS:
    checkpoint = torch.load(CONFIG['output_dir'] + "/best_model.pt", map_location=device)
    model.load_state_dict(checkpoint["model_state_dict"])
    model.eval()
    item_map = {(it["date"], it["region"]): it for it in test_dataset.items}

for date, region in TARGETS:
    if (date, region) not in available:
        print(f"[警告] test set 中沒有 {date}/{region}，跳過")
        continue

    item = item_map[(date, region)]
    P = test_dataset.patch_size
    canvas = np.full((item["padded_h"], item["padded_w"]),
                     float(CONFIG["nodata_value"]), dtype=np.float32)

    idxs = [i for i, s in enumerate(test_dataset.samples)
            if s["date"] == date and s["region"] == region]
    with torch.no_grad():
        for k in range(0, len(idxs), CONFIG["batch_size"]):
            batch_idx = idxs[k:k + CONFIG["batch_size"]]
            pairs_xy = [(test_dataset[i][0], test_dataset[i][1]) for i in batch_idx]
            x = torch.stack([p[0] for p in pairs_xy]).to(device)
            probs = torch.sigmoid(model(x)).squeeze(1).cpu().numpy()
            for bi, i in enumerate(batch_idx):
                s = test_dataset.samples[i]
                pred = (probs[bi] >= 0.5).astype(np.float32)
                y_np = pairs_xy[bi][1].numpy()
                pred[y_np == CONFIG["nodata_value"]] = CONFIG["nodata_value"]
                canvas[s["row"]:s["row"] + P, s["col"]:s["col"] + P] = pred
    pred_full = canvas[:item["original_h"], :item["original_w"]]

    gt = np.load(os.path.join(CONFIG["xy_dir"], "y", date, f"{region}.npy")).astype(np.float32)
    if gt.ndim == 3:
        gt = gt[0]  # (1, H, W) → (H, W)

    SAVE_PATH = os.path.join(CONFIG['output_dir'], f"confusion_map_{date}_{region}.png")
    counts = myplot.plot_confusion_map(
        pred_path=pred_full,
        gt_path=gt,
        ignore_index=CONFIG['nodata_value'],
        save_path=SAVE_PATH,
        title=f"Confusion Map for {date}/{region}",
    )
    print(counts)